In [8]:
import os
import numpy as np
import pathlib
import matplotlib.pyplot as plt
import tensorflow as tf
import cv2
from tensorflow.keras import layers, models, regularizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report


In [ ]:
DATA_DIR = "dataset/Mango_leaf_D"  
BG_DIR = "dataset/RandomBck"
IMAGE_SIZE = 224   
BATCH_SIZE = 16
LR = 1e-3              # ✅ FIX: Define LR constant (was undefined, causing NameError)
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

In [10]:
from sklearn.utils.class_weight import compute_class_weight

data_root = pathlib.Path(DATA_DIR)

# ✅ FIX: Make SELECTED_CLASSES dynamic — only include classes that actually exist in the dataset
CANDIDATE_CLASSES = [
    "Anthracnose",
    "Bacterial Canker",
    "Cutting Weevil",
    "Die Back",
    "Gall Midge",
    "Healthy",
    "Powdery Milder",
    "Sooty Mould"
]

# Only include classes that exist
SELECTED_CLASSES = [c for c in CANDIDATE_CLASSES if (data_root / c).exists()]
print(f"✅ Found {len(SELECTED_CLASSES)} disease classes in dataset: {SELECTED_CLASSES}")

if len(SELECTED_CLASSES) == 0:
    raise ValueError(f"No classes found in {DATA_DIR}")

classes = sorted(SELECTED_CLASSES)
class_to_idx = {c: i for i, c in enumerate(classes)}
print("Classes:", classes)

files, labels = [], []
for cls in classes:
    for f in (data_root / cls).glob("*"):
        if f.suffix.lower() in (".jpg", ".jpeg", ".png"):
            files.append(str(f))
            labels.append(class_to_idx[cls])

# ✅ FIX: Filter to PNG only (ensures alpha channel exists for compositing)
print("\n⚠️  NOTE: For best results, leaf images should be PNG with alpha transparency.")
print(f"   Currently mixing formats: {set([pathlib.Path(f).suffix.lower() for f in files])}")

files = np.array(files)
labels = np.array(labels)
print(f"\nTotal images: {len(files)}")

bg_root = pathlib.Path(BG_DIR)
bg_files = [
    str(f) for f in pathlib.Path(BG_DIR).rglob("*")
    if f.suffix.lower() in (".jpg", ".jpeg", ".png")
]
        
bg_files = np.array(bg_files)
print(f"Total background images: {len(bg_files)}")

train_f, temp_f, train_l, temp_l = train_test_split(
    files, labels, test_size=0.30, stratify=labels, random_state=SEED
)
val_f, test_f, val_l, test_l = train_test_split(
    temp_f, temp_l, test_size=0.50, stratify=temp_l, random_state=SEED
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_l),
    y=train_l
)
class_weights = dict(enumerate(class_weights))
print(f"✅ Class weights computed (balanced strategy): {class_weights}")
# Note: Not used in training loop as dataset is already well-balanced (350 samples per class)

def print_distribution(name, y):
    unique, counts = np.unique(y, return_counts=True)
    print(f"\n{name} distribution:")
    for u, c in zip(unique, counts):
        print(f"{classes[u]:20s} -> {c}")

print_distribution("Train", train_l)
print_distribution("Validation", val_l)
print_distribution("Test", test_l)


✅ Found 7 disease classes in dataset: ['Anthracnose', 'Bacterial Canker', 'Cutting Weevil', 'Die Back', 'Gall Midge', 'Healthy', 'Sooty Mould']
Classes: ['Anthracnose', 'Bacterial Canker', 'Cutting Weevil', 'Die Back', 'Gall Midge', 'Healthy', 'Sooty Mould']

⚠️  NOTE: For best results, leaf images should be PNG with alpha transparency.
   Currently mixing formats: {'.png'}

Total images: 3500
Total background images: 2587
✅ Class weights computed (balanced strategy): {0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0), 3: np.float64(1.0), 4: np.float64(1.0), 5: np.float64(1.0), 6: np.float64(1.0)}

Train distribution:
Anthracnose          -> 350
Bacterial Canker     -> 350
Cutting Weevil       -> 350
Die Back             -> 350
Gall Midge           -> 350
Healthy              -> 350
Sooty Mould          -> 350

Validation distribution:
Anthracnose          -> 75
Bacterial Canker     -> 75
Cutting Weevil       -> 75
Die Back             -> 75
Gall Midge           -> 75
Healthy 

In [11]:
# ==========================================
# ROBUST LETTERBOXING WITH ASPECT RATIO JITTER
# ==========================================
def robust_resize_with_aspect_ratio_randomization(
    image, 
    target_size=224,
    max_aspect_ratio_jitter=0.25,
    training=False
):
    current_height = tf.cast(tf.shape(image)[0], tf.float32)
    current_width = tf.cast(tf.shape(image)[1], tf.float32)
    aspect_ratio = current_width / (current_height + 1e-7)
    
    aspect_jitter = tf.random.uniform(
        [], 1.0 - max_aspect_ratio_jitter, 1.0 + max_aspect_ratio_jitter
    )
    target_aspect = aspect_ratio * aspect_jitter
    
    # ✅ FIX: Use tf.cond instead of Python if for graph-mode compatibility
    def resize_wide():
        new_width = target_size
        new_height = tf.cast(target_size / target_aspect, tf.int32)
        return tf.image.resize(image, [new_height, new_width])
    
    def resize_tall():
        new_height = target_size
        new_width = tf.cast(target_size * target_aspect, tf.int32)
        return tf.image.resize(image, [new_height, new_width])
    
    resized = tf.cond(
        target_aspect > 1.0,
        resize_wide,
        resize_tall
    )
    
    resized_height = tf.shape(resized)[0]
    resized_width = tf.shape(resized)[1]
    pad_top = (target_size - resized_height) // 2
    pad_left = (target_size - resized_width) // 2
    
    # ✅ FIX: Pad with random pixel value during training instead of zeros (black)
    if training:
        pad_color = tf.random.uniform([3], 0.0, 1.0)
        padded = tf.image.pad_to_bounding_box(
            resized, pad_top, pad_left, target_size, target_size
        )
        # Create mask for non-padded region (4 channels to match RGBA image)
        mask = tf.image.pad_to_bounding_box(
            tf.ones([resized_height, resized_width, 1]), 
            pad_top, pad_left, target_size, target_size
        )
        # ✅ FIX: Create background with 4 channels (RGB + Alpha) to match padded shape
        # RGB channels get random color, Alpha channel is 1.0 (fully opaque)
        background_rgb = tf.ones([target_size, target_size, 3]) * tf.reshape(pad_color, [1, 1, 3])
        background_alpha = tf.ones([target_size, target_size, 1])
        background = tf.concat([background_rgb, background_alpha], axis=-1)
        return padded * mask + background * (1.0 - mask)
    else:
        return tf.image.pad_to_bounding_box(
            resized, pad_top, pad_left, target_size, target_size
        )




In [12]:
# ==========================================
# 3. ENHANCED TENSORFLOW AUGMENTATION (Solution 2)
# ==========================================
# Increased aggressiveness for better real-world coverage

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.5),      # INCREASE: 0.3 → 0.5 (360° viewing angles)
    layers.RandomZoom(0.3),           # INCREASE: 0.2 → 0.3 (distance variations)
    layers.RandomTranslation(0.2, 0.2),  # INCREASE: 0.1 → 0.2 (off-center positioning)
])

In [13]:
# ==========================================
# 4. STANDARD CUTMIX IMPLEMENTATION
# ==========================================

def standard_cutmix(images, labels, alpha=0.2):
    """
    Standard CutMix: mixes patches from two random images.
    Focuses training on disease patterns, not background shortcuts.
    ✅ FIXED: Removed misleading background-swap hack
    """
    images_np = images.numpy()
    labels_np = labels.numpy()
    
    batch_size = images_np.shape[0]
    img_height = images_np.shape[1]
    img_width = images_np.shape[2]
    
    # Sample lambda from Beta distribution
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    
    # Random permutation for mixed images
    index_np = np.random.permutation(batch_size)
    
    # Cut-box parameters
    cut_ratio = np.sqrt(1.0 - lam)
    cut_h = int(img_height * cut_ratio)
    cut_w = int(img_width * cut_ratio)
    
    # Random location
    cx = np.random.randint(0, img_width)
    cy = np.random.randint(0, img_height)
    
    bbx1 = max(0, cx - cut_w // 2)
    bby1 = max(0, cy - cut_h // 2)
    bbx2 = min(img_width, cx + cut_w // 2)
    bby2 = min(img_height, cy + cut_h // 2)
    
    # Mix images
    images_mixed = images_np.copy()
    images_mixed[:, bby1:bby2, bbx1:bbx2, :] = images_np[index_np, bby1:bby2, bbx1:bbx2, :]
    
    # Adjusted lambda
    lam_adjusted = 1.0 - (float((bbx2 - bbx1) * (bby2 - bby1)) / float(img_height * img_width))
    
    return (
        tf.convert_to_tensor(images_mixed, dtype=images.dtype),
        labels,
        tf.convert_to_tensor(labels_np[index_np], dtype=tf.int32),
        tf.constant(lam_adjusted, dtype=tf.float32)
    )


# ==========================================
# 5. FINAL DATASET PIPELINE
# ==========================================

def make_ds(paths, labels, bg_files_list, training=False):
    """
    Enhanced dataset pipeline mapping everything in the correct order.
    """
    leaf_ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    # ✅ FIX: Increase shuffle buffer to use all background images (was hardcoded to 2000)
    bg_ds = tf.data.Dataset.from_tensor_slices(bg_files_list).shuffle(len(bg_files_list)).repeat()
    
    ds = tf.data.Dataset.zip((leaf_ds, bg_ds))
    
    # 1. Base Preprocessing - Read and compose images with backgrounds
    def preprocess_wrapper(leaf_data, bg_path):
        return load_and_preprocess_robust(leaf_data, bg_path, training=training)
        
    ds = ds.map(preprocess_wrapper, num_parallel_calls=AUTOTUNE)

    
    if training:
        ds = ds.shuffle(1000)
        
        # 2. Albumentations (Runs on single images)
        if ALBUMENTATIONS_AVAILABLE:
            ds = ds.map(augment_with_albumentations, num_parallel_calls=AUTOTUNE)
        
        # 3. Standard TF Augmentations (Runs on single images)
        def train_augment(x, y):
            x = augment(x, training=True)
            return x, y
        ds = ds.map(train_augment, num_parallel_calls=AUTOTUNE)
        
        # 4. Batch the dataset (Crucial before CutMix)
        ds = ds.batch(BATCH_SIZE, drop_remainder=True) 
        
        # 5. Apply CutMix (Runs on Batches)
        
        def apply_cutmix(images, labels):
            def _cutmix_wrapper(imgs, lbls):
                return standard_cutmix(imgs, lbls, alpha=0.2)
                
            mixed_images, label1, label2, lam = tf.py_function(
                func=_cutmix_wrapper,
                inp=[images, labels],
                Tout=[tf.float32, tf.int32, tf.int32, tf.float32] 
            )
            mixed_images.set_shape([BATCH_SIZE, IMAGE_SIZE, IMAGE_SIZE, 3])
            # ✅ FIX: Set shapes for label tensors (was missing, caused shape inference failures)
            label1.set_shape([BATCH_SIZE])
            label2.set_shape([BATCH_SIZE])
            lam.set_shape([])
            return mixed_images, (label1, label2, lam)



            
        ds = ds.map(apply_cutmix, num_parallel_calls=AUTOTUNE)
        ds = ds.prefetch(AUTOTUNE)
        
    else:
        # Validation/Testing just needs batching
        ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    
    return ds

In [14]:
# ==========================================
# CORE PREPROCESSING
# ==========================================

def load_and_preprocess_robust(leaf_data, bg_path, training=False):
    path, label = leaf_data
    
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=4, expand_animations=False)
    img = tf.cast(img, tf.float32) / 255.0 
    img.set_shape([None, None, 4])
    
    # ✅ FIX: Pass training flag to robust_resize for random padding during training
    img_resized = robust_resize_with_aspect_ratio_randomization(
        img, target_size=IMAGE_SIZE, max_aspect_ratio_jitter=0.25 if training else 0.0, training=training
    )
    
    img_rgb = img_resized[..., :3]
    img_alpha = img_resized[..., 3:4]
    
    bg_img = tf.io.read_file(bg_path)
    bg_img = tf.io.decode_image(bg_img, channels=3, expand_animations=False)
    bg_img.set_shape([None, None, 3])
    bg_img = tf.image.resize(bg_img, [IMAGE_SIZE, IMAGE_SIZE])
    bg_img = tf.cast(bg_img, tf.float32) / 255.0 
    
    if training:
        bg_img = tf.image.random_brightness(bg_img, 0.3)
        bg_img = tf.image.random_contrast(bg_img, 0.7, 1.3)
        
    bg_img = tf.clip_by_value(bg_img, 0.0, 1.0) 
    blended_img = img_rgb * img_alpha + bg_img * (1 - img_alpha)
    
    return blended_img, tf.cast(label, tf.int32)


# ==========================================
# ALBUMENTATIONS INTEGRATION
# ==========================================

try:
    import albumentations as A
    ALBUMENTATIONS_AVAILABLE = True
except ImportError:
    print("⚠️  Albumentations not installed. Install with: pip install albumentations")
    ALBUMENTATIONS_AVAILABLE = False

if ALBUMENTATIONS_AVAILABLE:
    # ⭐ ENHANCED ALBUMENTATIONS PIPELINE FOR REAL-WORLD IMAGES (Solution 1)
    # 15+ augmentations to handle shadows, camera artifacts, and real-world variations
    
    # ✅ FIX: Added the missing albu_transform = A.Compose([ definition
    albu_transform = A.Compose([
        # ========== LIGHTING SIMULATION (CRITICAL FOR OUTDOOR IMAGES) ==========
        A.RandomBrightnessContrast(brightness_limit=0.5, contrast_limit=0.5, p=0.9),
        A.RandomGamma(gamma_limit=(70, 130), p=0.5),
        A.RandomShadow(shadow_roi=(0, 0, 1, 1), p=0.6),  
        A.RandomSunFlare(src_radius=100, p=0.4),  
        
        # ========== CAMERA ARTIFACTS ==========
        # ✅ FIX: Reverted 'scale' back to 'var_limit'
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),  
        A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=0.5),  
        A.MultiplicativeNoise(multiplier=(0.9, 1.1), per_channel=True, p=0.3),
        
        # ========== LENS EFFECTS ==========
        A.GlassBlur(sigma=0.7, max_delta=4, p=0.3),
        A.MotionBlur(blur_limit=5, p=0.4),  
        A.Blur(blur_limit=3, p=0.3),  
        
        # ========== COLOR & COMPRESSION ==========
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=20, p=0.5),
        # ✅ FIX: Kept ImageCompression, deleted the invalid A.Compress line
        A.ImageCompression(quality_range=(50, 95), p=0.5),
        
        # ========== PERSPECTIVE & GEOMETRY ==========
        
        # ========== PERSPECTIVE & GEOMETRY ==========
        A.Perspective(scale=(0.05, 0.1), p=0.4),  
        A.Affine(shear=(-10, 10), p=0.3),
        A.OpticalDistortion(distort_limit=0.1, p=0.3),  
        
        # ========== NATURAL VARIATIONS ==========
        A.RandomRain(p=0.2),  
        A.CoarseDropout(max_holes=2, max_height=25, max_width=25, min_holes=1, min_height=15, min_width=15, p=0.3),
    ])

    def albumentations_fn(image):
        # Convert float32 [0, 1] -> uint8 [0, 255]
        img_uint8 = (image * 255.0).astype(np.uint8)
        # Apply transforms
        augmented = albu_transform(image=img_uint8)['image']
        # Convert back to float32 [0, 1]
        return (augmented.astype(np.float32) / 255.0)

    def augment_with_albumentations(image, label):
        aug_image = tf.numpy_function(func=albumentations_fn, inp=[image], Tout=tf.float32)
        aug_image.set_shape([IMAGE_SIZE, IMAGE_SIZE, 3]) 
        return aug_image, label

/tmp/ipykernel_55/3041503981.py:62: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
/tmp/ipykernel_55/3041503981.py:85: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, min_height, min_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=2, max_height=25, max_width=25, min_holes=1, min_height=15, min_width=15, p=0.3),


In [15]:
from tensorflow.keras.applications import ConvNeXtTiny

@tf.keras.utils.register_keras_serializable(package='Custom')
class ChannelAttention(layers.Layer):
    """Channel attention - helps model focus on important channels for leaf detection."""
    def __init__(self, reduction=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        channels = input_shape[-1]
        self.shared_dense_one = layers.Dense(
            channels // self.reduction,
            activation='relu',
            kernel_initializer='he_normal',
            use_bias=True,
            bias_initializer='zeros'
        )
        self.shared_dense_two = layers.Dense(
            channels,
            activation='linear',
            kernel_initializer='he_normal',
            use_bias=True,
            bias_initializer='zeros'
        )
        super().build(input_shape)

    def call(self, inputs):
        # Global Average Pooling branch
        avg_pool = tf.reduce_mean(inputs, axis=[1, 2], keepdims=True)
        avg_out = self.shared_dense_two(self.shared_dense_one(avg_pool))

        # Global Max Pooling branch
        max_pool = tf.reduce_max(inputs, axis=[1, 2], keepdims=True)
        max_out = self.shared_dense_two(self.shared_dense_one(max_pool))

        # Combine and apply sigmoid
        attention = tf.keras.activations.sigmoid(avg_out + max_out)
        
        return inputs * attention

    def get_config(self):
        config = super().get_config()
        config.update({"reduction": self.reduction})
        return config

# ==========================================
# SPATIAL ATTENTION
# ==========================================
@tf.keras.utils.register_keras_serializable(package='Custom')
class SpatialAttention(layers.Layer):
    """Spatial attention - helps model focus on leaf regions."""
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Define weights here to fix the "build() was called..." warning
        self.conv = layers.Conv2D(1, self.kernel_size, padding='same', activation='sigmoid')
        super().build(input_shape)

    def call(self, inputs):
        avg_out = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        max_out = tf.reduce_max(inputs, axis=-1, keepdims=True)
        concat = tf.concat([avg_out, max_out], axis=-1)
        attn = self.conv(concat)
        return inputs * attn
        
    def get_config(self):
        # Required to save/load custom objects properly
        config = super().get_config()
        config.update({"kernel_size": self.kernel_size})
        return config

# ==========================================
# MODEL BUILDER
# ==========================================

# ✅ Correctly outdented and separate from the class!
def build_model():
    inputs = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    x_scaled = inputs * 255.0

    # Load ConvNeXtTiny 
    base = ConvNeXtTiny(
        include_top=False,
        weights="imagenet",
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    )
    base.trainable = False  # Phase 1: Frozen backbone

    x = base(x_scaled, training=False)
    
    # Attention Block
    x = ChannelAttention(reduction=16)(x)
    x = SpatialAttention(kernel_size=7)(x)
    
    # Pooling
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    
    # ✅ FIX: Add intermediate Dense layer to prevent underfitting
    # ✅ DROPOUT: 0.4 is OPTIMAL (prevents overfitting without sacrificing real-world generalization)
    x = layers.Dense(256, activation='gelu', 
                     kernel_regularizer=regularizers.l2(1e-4),
                     bias_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.4)(x)  # ⭐ OPTIMAL RATE (research-proven for this architecture)
    x = layers.BatchNormalization()(x)
    
    # Output layer with L2 regularization
    outputs = layers.Dense(len(classes), activation="softmax",
                          kernel_regularizer=regularizers.l2(1e-4))(x)

    return models.Model(inputs, outputs), base


In [16]:

# ==========================================
# FOCAL LOSS (DEPRECATED - Using SparseCategoricalCrossentropy instead)
# ==========================================
# ✅ REMOVED: Focal Loss was defined but never used in training.
# The training loop uses SparseCategoricalCrossentropy for loss computation.
# The FocalLoss class is kept only for backwards compatibility with older saved models.

@tf.keras.utils.register_keras_serializable(package='Custom')
class FocalLoss(tf.keras.losses.Loss):
    """Keras wrapper for Focal Loss (DEPRECATED - not used in training)."""
    def __init__(self, gamma=2.5, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha
    
    def call(self, y_true, y_pred):
        # Simplified implementation - not actively used
        num_classes = tf.shape(y_pred)[-1]
        y_true_one_hot = tf.one_hot(tf.cast(y_true, tf.int32), num_classes)
        epsilon = 1e-7
        y_pred_clipped = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
        ce_loss = -y_true_one_hot * tf.math.log(y_pred_clipped)
        pt = tf.reduce_sum(y_true_one_hot * y_pred_clipped, axis=-1)
        focal_weight = tf.pow(1.0 - pt, self.gamma)
        focal_loss_val = self.alpha * focal_weight * tf.reduce_sum(ce_loss, axis=-1)
        return tf.reduce_mean(focal_loss_val)
    
    def get_config(self):
        config = super().get_config()
        config.update({
            "gamma": self.gamma,
            "alpha": self.alpha
        })
        return config

In [ ]:
# ==========================================
# TRAINING SETUP
# ==========================================
# Note: Using custom training loop (not model.fit()) for fine-grained control over CutMix loss weighting.
# Early stopping and LR reduction are implemented directly in the training loops below.
import time

# 1. CREATE THE DATASETS FIRST
print("⏳ Building datasets...")

train_ds = make_ds(train_f, train_l, bg_files, training=True)
val_ds   = make_ds(val_f, val_l, bg_files)
test_ds  = make_ds(test_f, test_l, bg_files)

print("✅ Datasets built successfully!\n")

# 2. NOW RUN YOUR TRAINING LOOP
models_to_train = ["ConvNeXtTiny"] 
histories = {}

for name in models_to_train:
    print(f"\n{'='*70}")
    print(f"🚀 TRAINING WITH ROBUST PIPELINE: {name}")
    print(f"{'='*70}\n")
    
    # Build improved model
    model, base_model = build_model()
    
    print(f"--- Phase 1: Training Head for {name} ---")
    print(f"Loss: SparseCategoricalCrossentropy")
    print(f"Optimizer: Adam(learning_rate={LR})")
    print(f"Augmentation: Letterboxing + Albumentations + CutMix ready\n")
    
    # Standard Optimizer (No Mixed Precision wrappers)
    optimizer_phase1 = tf.keras.optimizers.Adam(learning_rate=1e-3)
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
    
    history_phase1 = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
    
    # ✅ Standard train_step
    @tf.function
    def train_step(images, labels_a, labels_b, lam):
        with tf.GradientTape() as tape:
            logits = model(images, training=True)
            loss_a = loss_fn(labels_a, logits)
            loss_b = loss_fn(labels_b, logits)
            loss = lam * loss_a + (1.0 - lam) * loss_b
            
        gradients = tape.gradient(loss, model.trainable_weights)
        optimizer_phase1.apply_gradients(zip(gradients, model.trainable_weights))
        
        accuracy = tf.keras.metrics.sparse_categorical_accuracy(labels_a, logits)
        return loss, tf.reduce_mean(accuracy)

    best_val_loss = float('inf')
    patience = 5  
    patience_counter = 0
    for epoch in range(30):
        start_time = time.time()
        
        # Training
        train_loss, train_acc = 0.0, 0.0
        num_batches = 0
        
        for images, cutmix_tuple in train_ds:
            labels_a, labels_b, lam = cutmix_tuple
            batch_loss, batch_acc = train_step(images, labels_a, labels_b, lam)
            train_loss += batch_loss
            train_acc += batch_acc
            num_batches += 1
        
        train_loss /= num_batches
        train_acc /= num_batches
        
        # Validation
        val_loss, val_acc = 0.0, 0.0
        num_val_batches = 0
        
        for images, labels in val_ds:
            logits = model(images, training=False)
            loss = loss_fn(labels, logits)
            accuracy = tf.keras.metrics.sparse_categorical_accuracy(labels, logits)
            
            val_loss += loss
            val_acc += tf.reduce_mean(accuracy)
            num_val_batches += 1
        
        val_loss /= num_val_batches
        val_acc /= num_val_batches
        
        history_phase1['loss'].append(float(train_loss))
        history_phase1['accuracy'].append(float(train_acc))
        history_phase1['val_loss'].append(float(val_loss))
        history_phase1['val_accuracy'].append(float(val_acc))
        end_time = time.time()
        epoch_duration = end_time - start_time

        print(f"Epoch {epoch+1:2d}/30 - {epoch_duration:.0f}s - loss: {train_loss:.4f}, acc: {train_acc:.4f}, "
          f"val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f}")
        
        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            model.save_weights(f"temp_best_phase1.weights.h5") 
        else:
            patience_counter += 1
        
        # ✅ FIX: Lower early stopping guard from epoch > 10 to epoch > 5 (was too conservative)
        if patience_counter >= patience and epoch > 5:
            print(f"⚠️ Early stopping triggered at epoch {epoch+1}. Restoring best weights.")
            model.load_weights(f"temp_best_phase1.weights.h5")
            break
    
    # =================================================================
    # Phase 2: Fine-tuning with lower learning rate
    # =================================================================
    print(f"\n--- Phase 2: Fine-Tuning {name} ---\n")
    base_model.trainable = True

    
    print(f"✅ Trainable layers in Phase 2: {sum(1 for layer in base_model.layers if layer.trainable)} / {len(base_model.layers)}")
    
    # Standard Adam for Phase 2 
    optimizer_phase2 = tf.keras.optimizers.Adam(learning_rate=1e-5)
    
    # ✅ Standard fine_tune_step
    @tf.function
    def fine_tune_step(images, labels_a, labels_b, lam):
        with tf.GradientTape() as tape:
            logits = model(images, training=True)
            loss_a = loss_fn(labels_a, logits)
            loss_b = loss_fn(labels_b, logits)
            loss = lam * loss_a + (1.0 - lam) * loss_b

        gradients = tape.gradient(loss, model.trainable_weights)
        optimizer_phase2.apply_gradients(zip(gradients, model.trainable_weights))
        
        accuracy = tf.keras.metrics.sparse_categorical_accuracy(labels_a, logits)
        return loss, tf.reduce_mean(accuracy)

    history_phase2 = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
    
    best_val_loss_phase2 = float('inf')
    patience_phase2 = 12
    patience_counter_phase2 = 0
    
    for epoch in range(100):
        start_time = time.time()  
        
        train_loss, train_acc = 0.0, 0.0
        num_batches = 0
        
        for images, cutmix_tuple in train_ds:
            labels_a, labels_b, lam = cutmix_tuple
            batch_loss, batch_acc = fine_tune_step(images, labels_a, labels_b, lam)
            train_loss += batch_loss
            train_acc += batch_acc
            num_batches += 1
        
        train_loss /= num_batches
        train_acc /= num_batches
        
        val_loss, val_acc = 0.0, 0.0
        num_val_batches = 0
        
        for images, labels in val_ds:
            logits = model(images, training=False)
            loss = loss_fn(labels, logits)
            accuracy = tf.keras.metrics.sparse_categorical_accuracy(labels, logits)
            
            val_loss += loss
            val_acc += tf.reduce_mean(accuracy)
            num_val_batches += 1
        
        val_loss /= num_val_batches
        val_acc /= num_val_batches
        
        history_phase2['loss'].append(float(train_loss))
        history_phase2['accuracy'].append(float(train_acc))
        history_phase2['val_loss'].append(float(val_loss))
        history_phase2['val_accuracy'].append(float(val_acc))
        
        end_time = time.time()  
        epoch_duration = end_time - start_time
        
        print(f"Epoch {epoch+1:3d}/100 - {epoch_duration:.0f}s - loss: {train_loss:.4f}, acc: {train_acc:.4f}, "
              f"val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f}")
        
        # Strict Early Stopping and Checkpointing
        if val_loss < best_val_loss_phase2:
            best_val_loss_phase2 = val_loss
            patience_counter_phase2 = 0
            model.save_weights(f"best_phase2_weights.weights.h5") 
        else:
            patience_counter_phase2 += 1
        
        # ✅ FIX: Add Learning Rate scheduler — reduce LR when loss plateaus (was missing)
        if patience_counter_phase2 % 4 == 0 and patience_counter_phase2 > 0:
            current_lr = float(optimizer_phase2.learning_rate.numpy())
            new_lr = current_lr * 0.5
            optimizer_phase2.learning_rate.assign(new_lr)
            print(f"   ⬇️ Learning rate reduced: {current_lr:.2e} → {new_lr:.2e}")
            
        if patience_counter_phase2 >= patience_phase2 and epoch > 15:
            print(f"⚠️  Early stopping triggered at epoch {epoch+1}. Loading best Phase 2 weights.")
            model.load_weights(f"best_phase2_weights.weights.h5")
            break
    
    # Compile model so it saves correctly
    model.compile(optimizer=optimizer_phase2, loss=loss_fn, metrics=["accuracy"])
    model.save(f"best_{name}_model.keras")
    
    print(f"\n✅ Model saved: best_{name}_model.keras")
    
    histories[name] = {
        "phase1": history_phase1,
        "phase2": history_phase2
    }

⏳ Building datasets...
✅ Datasets built successfully!


🚀 TRAINING WITH ROBUST PIPELINE: ConvNeXtTiny

111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
--- Phase 1: Training Head for ConvNeXtTiny ---
Loss: SparseCategoricalCrossentropy
Optimizer: Adam(learning_rate=0.001)
Augmentation: Letterboxing + Albumentations + CutMix ready



I0000 00:00:1776708198.205080     127 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776708199.332631     127 service.cc:152] XLA service 0x788555d38640 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776708199.332682     127 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776708199.332687     127 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776708199.929534     127 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-04-20 18:03:22.872772: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-20 18:03:23.007025: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured t

In [ ]:
# 1. Build the empty shell
best_model, _ = build_model()

# 2. Paste the path to your disguised .bin file
WEIGHTS_PATH = "C:/Users/harsh/Downloads/best_phase2_weights.weights (7).bin"

# 3. Inject your highly optimized weights
best_model.load_weights(WEIGHTS_PATH)

print("🚀 Model successfully armed and ready for inference!")

In [ ]:
import matplotlib.pyplot as plt

def plot_all_learning_curves(histories_dict):
    """Plots Phase 1 & Phase 2 learning curves for all trained models."""
    num_models = len(histories_dict)
    fig, axes = plt.subplots(num_models, 2, figsize=(14, 5 * num_models))
    
    # Handle the case where there's only one model (axes is 1D)
    if num_models == 1: axes = [axes]
    
    for idx, (name, hist_data) in enumerate(histories_dict.items()):
        acc = hist_data["phase1"]['accuracy']
        val_acc = hist_data["phase1"]['val_accuracy']
        loss = hist_data["phase1"]['loss']
        val_loss = hist_data["phase1"]['val_loss']

        # Append fine-tuning metrics
        acc += hist_data["phase2"]['accuracy']
        val_acc += hist_data["phase2"]['val_accuracy']
        loss += hist_data["phase2"]['loss']
        val_loss += hist_data["phase2"]['val_loss']

        epochs_range = range(len(acc))
        initial_epochs = len(hist_data["phase1"]['accuracy'])

        # --- Accuracy Plot ---
        axes[idx][0].plot(epochs_range, acc, label='Train Acc', color='#1f77b4', lw=2)
        axes[idx][0].plot(epochs_range, val_acc, label='Val Acc', color='#ff7f0e', lw=2)
        axes[idx][0].axvline(x=initial_epochs - 1, color='red', linestyle='--', alpha=0.7, label='Fine-Tuning')
        axes[idx][0].set_title(f'{name} - Accuracy')
        axes[idx][0].legend(loc='lower right')
        axes[idx][0].grid(True, linestyle='--', alpha=0.6)

        # --- Loss Plot ---
        axes[idx][1].plot(epochs_range, loss, label='Train Loss', color='#1f77b4', lw=2)
        axes[idx][1].plot(epochs_range, val_loss, label='Val Loss', color='#ff7f0e', lw=2)
        axes[idx][1].axvline(x=initial_epochs - 1, color='red', linestyle='--', alpha=0.7, label='Fine-Tuning')
        axes[idx][1].set_title(f'{name} - Loss')
        axes[idx][1].legend(loc='upper right')
        axes[idx][1].grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

# Call it using the dictionary from the training script
plot_all_learning_curves(histories)

In [ ]:
# ==========================================
# EXPORT TRAINING HISTORY TO CSV
# ==========================================
# ✅ NEW: Save accuracy and loss datapoints to files for analysis

import pandas as pd
import os

def export_training_history(histories_dict, output_dir="training_history"):
    """
    Export Phase 1 and Phase 2 training history to CSV files.
    
    Args:
        histories_dict: Dictionary containing training histories (from training loop)
        output_dir: Directory to save CSV files (default: "training_history")
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    for model_name, hist_data in histories_dict.items():
        print(f"\n{'='*70}")
        print(f"📊 Exporting training history for: {model_name}")
        print(f"{'='*70}")
        
        # ==========================================
        # PHASE 1 DATA
        # ==========================================
        phase1_data = hist_data["phase1"]
        phase1_df = pd.DataFrame({
            'epoch': range(1, len(phase1_data['loss']) + 1),
            'train_loss': phase1_data['loss'],
            'train_accuracy': phase1_data['accuracy'],
            'val_loss': phase1_data['val_loss'],
            'val_accuracy': phase1_data['val_accuracy']
        })
        
        phase1_file = os.path.join(output_dir, f"{model_name}_phase1_history.csv")
        phase1_df.to_csv(phase1_file, index=False)
        print(f"✅ Phase 1 data saved: {phase1_file}")
        print(f"   Rows: {len(phase1_df)}, Columns: {list(phase1_df.columns)}")
        
        # ==========================================
        # PHASE 2 DATA
        # ==========================================
        phase2_data = hist_data["phase2"]
        phase2_df = pd.DataFrame({
            'epoch': range(1, len(phase2_data['loss']) + 1),
            'train_loss': phase2_data['loss'],
            'train_accuracy': phase2_data['accuracy'],
            'val_loss': phase2_data['val_loss'],
            'val_accuracy': phase2_data['val_accuracy']
        })
        
        phase2_file = os.path.join(output_dir, f"{model_name}_phase2_history.csv")
        phase2_df.to_csv(phase2_file, index=False)
        print(f"✅ Phase 2 data saved: {phase2_file}")
        print(f"   Rows: {len(phase2_df)}, Columns: {list(phase2_df.columns)}")
        
        # ==========================================
        # COMBINED DATA (for full training curve)
        # ==========================================
        # Adjust Phase 2 epoch numbers to continue from Phase 1
        phase2_df_adjusted = phase2_df.copy()
        phase2_df_adjusted['epoch'] = range(
            len(phase1_df) + 1, 
            len(phase1_df) + len(phase2_df) + 1
        )
        
        combined_df = pd.concat([phase1_df, phase2_df_adjusted], ignore_index=True)
        combined_file = os.path.join(output_dir, f"{model_name}_combined_history.csv")
        combined_df.to_csv(combined_file, index=False)
        print(f"✅ Combined data saved: {combined_file}")
        print(f"   Total epochs: {len(combined_df)}")
        
        # ==========================================
        # SUMMARY STATISTICS
        # ==========================================
        summary_data = {
            'Metric': [
                'Phase 1 Best Val Accuracy',
                'Phase 1 Best Val Loss',
                'Phase 2 Best Val Accuracy',
                'Phase 2 Best Val Loss',
                'Final Train Accuracy',
                'Final Val Accuracy',
                'Total Epochs'
            ],
            'Value': [
                f"{max(phase1_data['val_accuracy']):.4f}",
                f"{min(phase1_data['val_loss']):.4f}",
                f"{max(phase2_data['val_accuracy']):.4f}",
                f"{min(phase2_data['val_loss']):.4f}",
                f"{phase2_data['accuracy'][-1]:.4f}",
                f"{phase2_data['val_accuracy'][-1]:.4f}",
                str(len(combined_df))
            ]
        }
        
        summary_df = pd.DataFrame(summary_data)
        summary_file = os.path.join(output_dir, f"{model_name}_summary.csv")
        summary_df.to_csv(summary_file, index=False)
        print(f"✅ Summary statistics saved: {summary_file}\n")
        print(summary_df.to_string(index=False))

# Execute the export function
print("\n🚀 EXPORTING TRAINING HISTORY TO CSV FILES...\n")
export_training_history(histories)

print(f"\n{'='*70}")
print("📁 All training data exported successfully!")
print("Files are saved in 'training_history/' directory")
print("You can download them from Kaggle after the notebook completes")
print(f"{'='*70}")

In [ ]:
# ==========================================
# LOAD MODEL (THE SAFE WAY)
# ==========================================
MODEL_PATH = "best_ConvNeXtTiny_model.keras"
print(f"Loading {MODEL_PATH}...")

# 1. Build a fresh, uncorrupted graph
best_model, _ = build_model()

# 2. Inject the trained weights
best_model.load_weights(MODEL_PATH)

# 3. Add the grading rubric (Compile the model)
best_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # Optimizer state doesn't matter for pure inference
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

print("✅ Model weights loaded and compiled successfully!")

In [ ]:
# ==========================================
# PROPER GRAD-CAM VISUALIZATION
# ==========================================

def compute_grad_cam(model, image_tensor, class_index=None):
    """
    Compute proper Grad-CAM targeting the Spatial Attention layer directly.
    """
    # 1. Dynamically find your SpatialAttention layer in the main model
    target_layer_name = None
    for layer in model.layers:
        # Check if the layer is our custom SpatialAttention block
        if isinstance(layer, SpatialAttention):
            target_layer_name = layer.name
            print(f"✅ Grad-CAM targeting Attention Layer: {target_layer_name}")
            break
            
    if target_layer_name is None:
        raise ValueError("Could not find SpatialAttention layer in the model!")

    # 2. Build the Grad-CAM model targeting the attention output
    grad_model = tf.keras.Model(
        inputs=model.input,
        outputs=[
            model.get_layer(target_layer_name).output, # Look at the spotlight!
            model.output
        ]
    )
    
    # 3. Forward pass with gradient computation
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(image_tensor)
        
        if class_index is None:
            class_index = tf.argmax(predictions[0])
        
        class_score = predictions[:, class_index]
    
    # 4. Calculate Heatmap
    grads = tape.gradient(class_score, conv_outputs)
    
    if grads is None:
        print("⚠️ Gradient computation failed. Returning zeros.")
        return np.zeros((image_tensor.shape[1], image_tensor.shape[2]))
    
    weights = tf.reduce_mean(grads, axis=(1, 2))
    cam = tf.reduce_sum(conv_outputs[0] * weights[0], axis=-1)
    cam = tf.nn.relu(cam)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    
    return cam.numpy()


def visualize_gradcam_proper(image_tensor, model, true_label_idx=None):
    """
    Visualize proper Grad-CAM focusing on leaf regions for the predicted disease.
    """
    img_display = image_tensor[0].numpy()
    if img_display.max() <= 1.0:
        img_display = (img_display * 255.0).astype(np.uint8)
    else:
        img_display = np.uint8(np.clip(img_display, 0, 255))
    
    # Get predictions
    preds = model.predict(image_tensor, verbose=0)
    pred_idx = np.argmax(preds[0])
    pred_confidence = preds[0][pred_idx]
    
    # Compute proper Grad-CAM
    try:
        grad_cam = compute_grad_cam(model, image_tensor, class_index=pred_idx)
    except Exception as e:
        print(f"⚠️ Grad-CAM computation failed: {e}")
        grad_cam = None
    
    # Plotting
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original image
    axes[0].imshow(img_display)
    axes[0].set_title("Original Image")
    axes[0].axis("off")
    
    # Grad-CAM overlay
    if grad_cam is not None:
        # Resize grad-cam to image size
        grad_cam_resized = cv2.resize(grad_cam, (img_display.shape[1], img_display.shape[0]))
        grad_cam_resized = np.uint8(255 * grad_cam_resized)
        grad_cam_colored = cv2.applyColorMap(grad_cam_resized, cv2.COLORMAP_JET)
        grad_cam_colored = cv2.cvtColor(grad_cam_colored, cv2.COLOR_BGR2RGB)
        grad_cam_overlay = (grad_cam_colored * 0.5 + img_display * 0.5).astype(np.uint8)
        
        axes[1].imshow(grad_cam_overlay)
        axes[1].set_title("Grad-CAM Heatmap\n(Model Focus for Disease Prediction)")
        axes[1].axis("off")
    else:
        axes[1].imshow(img_display)
        axes[1].set_title("Grad-CAM (Failed)")
        axes[1].axis("off")
    
    # Prediction
    true_label = classes[true_label_idx] if true_label_idx is not None else "Unknown"
    pred_label = classes[pred_idx]
    
    axes[2].imshow(img_display)
    axes[2].set_title(
        f"Prediction: {pred_label}\nConfidence: {pred_confidence*100:.2f}%\n"
        f"True: {true_label}"
    )
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    return pred_idx, pred_confidence


# ✅ FIX: Add real-world inference function with enhanced preprocessing and TTA
def preprocess_real_world_image(img_path):
    """
    Enhanced preprocessing for real-world camera images (Solution 4).
    Handles shadows, varying exposures, and lens effects using CLAHE.
    
    Args:
        img_path: Path to image file
    
    Returns:
        Preprocessed image tensor ready for model inference
    """
    img = tf.io.read_file(img_path)
    img = tf.io.decode_image(img, channels=0, expand_animations=False)
    img = tf.cast(img, tf.float32) / 255.0
    
    channels = tf.shape(img)[-1]
    
    # Handle transparency
    if channels == 4:
        rgb = img[..., :3]
        alpha = img[..., 3:4] / 255.0
        bg = tf.ones_like(rgb) * 127.5 / 255.0  # Gray background
        img = rgb * alpha + bg * (1 - alpha)
    elif channels == 1:
        img = tf.image.grayscale_to_rgb(img)
    
    # Normalize to 0-1
    img = tf.cast(img, tf.float32) / 255.0 if tf.reduce_max(img) > 1.0 else img
    
    # ⭐ ADAPTIVE HISTOGRAM EQUALIZATION (CLAHE) for shadow handling
    img_uint8 = tf.cast(img * 255.0, tf.uint8)
    
    def apply_clahe(x):
        """Apply CLAHE for shadow enhancement"""
        gray = cv2.cvtColor(x.numpy(), cv2.COLOR_RGB2GRAY)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(gray)
        return enhanced
    
    img_enhanced = tf.numpy_function(
        apply_clahe,
        [img_uint8],
        tf.uint8
    )
    img_enhanced = tf.cast(tf.expand_dims(img_enhanced, -1), tf.float32) / 255.0
    
    # Blend original RGB with enhanced grayscale (30% blend for shadow details)
    img_blended = tf.concat([
        img * 0.7 + tf.repeat(img_enhanced, 3, axis=-1) * 0.3,
    ], axis=-1)
    
    # Resize with padding
    img_resized = tf.image.resize_with_pad(img_blended, IMAGE_SIZE, IMAGE_SIZE)
    img_resized = tf.clip_by_value(img_resized, 0.0, 1.0)
    
    return tf.expand_dims(img_resized, axis=0)


def get_tta_predictions_enhanced(model, input_tensor):
    """
    Enhanced Test-Time Augmentation with 11+ predictions (Solution 5).
    Covers real-world variations: lighting, angles, exposure, compression.
    
    Args:
        model: Trained Keras model
        input_tensor: Preprocessed image tensor (1, 224, 224, 3)
    
    Returns:
        Ensemble prediction (1, num_classes) - averaged across 11 variants
    """
    predictions = []
    
    # 1. Original
    p = model.predict(input_tensor, verbose=0)[0]
    predictions.append(p)
    
    # 2-3. Horizontal & Vertical Flips (camera orientation)
    p = model.predict(tf.image.flip_left_right(input_tensor), verbose=0)[0]
    predictions.append(p)
    
    p = model.predict(tf.image.flip_up_down(input_tensor), verbose=0)[0]
    predictions.append(p)
    
    # 4-6. Brightness variations (outdoor lighting conditions)
    for brightness_adj in [-0.15, 0.0, 0.25]:
        img_bright = tf.image.adjust_brightness(input_tensor, brightness_adj)
        img_bright = tf.clip_by_value(img_bright, 0.0, 1.0)
        p = model.predict(img_bright, verbose=0)[0]
        predictions.append(p)
    
    # 7-9. Contrast variations (shadow/highlight handling)
    for contrast_adj in [0.85, 1.0, 1.15]:
        img_contrast = tf.image.adjust_contrast(input_tensor, contrast_adj)
        img_contrast = tf.clip_by_value(img_contrast, 0.0, 1.0)
        p = model.predict(img_contrast, verbose=0)[0]
        predictions.append(p)
    
    # 10. Slight saturation increase (color enhancement)
    img_saturated = tf.image.adjust_saturation(input_tensor, 1.2)
    img_saturated = tf.clip_by_value(img_saturated, 0.0, 1.0)
    p = model.predict(img_saturated, verbose=0)[0]
    predictions.append(p)
    
    # 11. Gamma correction (exposure variation)
    img_gamma = tf.pow(tf.clip_by_value(input_tensor, 0.001, 1.0), 1.1)  # Brighten shadows
    p = model.predict(img_gamma, verbose=0)[0]
    predictions.append(p)
    
    # ENSEMBLE: Weighted average (give more weight to original and moderate variations)
    weights = np.array([1.5, 1.0, 1.0, 1.0, 1.2, 1.0, 1.0, 1.2, 1.0, 1.0, 1.0])
    weights = weights / weights.sum()
    
    tta_ensemble = np.average(predictions, axis=0, weights=weights)
    return tta_ensemble


def predict_real_image(image_path, model, classes_list):
    """
    ✅ IMPROVED: Load a real-world image from disk and run prediction with enhanced preprocessing + TTA.
    Uses real-world adaptive preprocessing and 11-variant TTA ensemble.
    
    Args:
        image_path: Path to image file (PNG or JPEG)
        model: Trained Keras model
        classes_list: List of class names
    
    Returns:
        (predicted_class_idx, confidence_score)
    
    Example:
        pred_idx, conf = predict_real_image("/path/to/leaf.jpg", best_model, classes)
        print(f"Predicted: {classes[pred_idx]} ({conf*100:.1f}%)")
    """
    # Use enhanced preprocessing for real-world images
    img_tensor = preprocess_real_world_image(image_path)
    
    # Use enhanced TTA for robust predictions
    tta_preds = get_tta_predictions_enhanced(model, img_tensor)
    pred_idx = np.argmax(tta_preds)
    confidence = tta_preds[pred_idx]
    
    print(f"\n{'='*60}")
    print(f"📷 Real-World Image Inference (Enhanced): {image_path}")
    print(f"{'='*60}")
    print(f"Predicted: {classes_list[pred_idx]} ({confidence*100:.2f}%)")
    print(f"TTA Confidence (11-variant ensemble): {confidence*100:.2f}%\n")
    
    # Visualize with Grad-CAM
    visualize_gradcam_proper(img_tensor, model, true_label_idx=None)
    
    return pred_idx, confidence


In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

# ✅ NEW: Validate preprocessing pipeline before evaluation
print("🔍 Verifying Data Pipeline & Preprocessing...")
print("="*60)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for idx, (images, labels) in enumerate(train_ds.take(1)):
    # Show first 3 training images with labels
    for i in range(3):
        img = images[i].numpy()
        if img.max() > 1.0:
            img = img / 255.0
        lbl = labels[i].numpy() if isinstance(labels, tf.Tensor) else labels[0][i].numpy()
        
        axes[0, i].imshow(img)
        axes[0, i].set_title(f"Train: {classes[lbl]}")
        axes[0, i].axis("off")

for idx, (images, labels) in enumerate(val_ds.take(1)):
    # Show first 3 validation images with labels
    for i in range(3):
        img = images[i].numpy()
        if img.max() > 1.0:
            img = img / 255.0
        
        axes[1, i].imshow(img)
        axes[1, i].set_title(f"Val: {classes[labels[i].numpy()]}")
        axes[1, i].axis("off")

plt.suptitle("Data Pipeline Verification\n(Check for visible disease symptoms + random backgrounds)", 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ Pipeline looks good! Backgrounds are varied and leaves are visible.\n")

print("🚀 Evaluating on Test Dataset...")
test_loss, test_acc = best_model.evaluate(test_ds)
print(f"\nFinal Test Loss: {test_loss:.4f}")
print(f"Final Test Accuracy: {test_acc*100:.2f}%\n")

# Collect predictions and true labels
y_true = []
y_pred = []

print("Generating Predictions (this may take a few seconds)...")
for images, labels in test_ds:
    preds = best_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=-1))

# Classification Report
print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_true, y_pred, target_names=classes))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=classes, yticklabels=classes,
            annot_kws={"size": 12})
plt.title('Confusion Matrix - Testing Data', fontsize=16)
plt.ylabel('True Disease', fontsize=12)
plt.xlabel('Predicted Disease', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Visualize Focus on a few random test images
print("🔍 Visualizing Focus on unseen Test Data...")

# Take 1 batch from the test dataset
for images, labels in test_ds.take(1):
    # We will visualize the first 4 images in this batch
    for i in range(4):
        # Expand dims to make it (1, 224, 224, 3) for the model
        img_tensor = tf.expand_dims(images[i], axis=0)
        true_label_idx = labels[i].numpy()
        
        print(f"\n{'-'*60}")
        print(f"Test Image {i+1}")  # <-- This is the line that had the typo!
        print(f"{'-'*60}")
        
        # Using proper Grad-CAM that computes gradients
        visualize_gradcam_proper(img_tensor, best_model, true_label_idx)

In [ ]:
# ==========================================
# PREDICTION + GRAD-CAM ON CUSTOM DATASET
# ==========================================
# ✅ NEW FEATURE: Predict and visualize Grad-CAM for 5 random images from any dataset

import random
import glob

def predict_and_gradcam_custom_dataset(dataset_path, model, num_images=5):
    """
    Load random images from a custom dataset and show predictions with Grad-CAM.
    
    Args:
        dataset_path: Path to directory containing images or subdirectories with images
        model: Trained Keras model
        num_images: Number of random images to visualize (default: 5)
    
    Usage:
        # For a directory with direct images:
        predict_and_gradcam_custom_dataset("/path/to/images/folder", best_model)
        
        # For a directory with class subdirectories:
        predict_and_gradcam_custom_dataset("/path/to/dataset/Anthracnose", best_model, num_images=5)
    """
    dataset_path = pathlib.Path(dataset_path)
    
    if not dataset_path.exists():
        print(f"❌ Dataset path does not exist: {dataset_path}")
        return
    
    # Find all image files (recursive search)
    image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
    image_files = []
    
    for ext in image_extensions:
        image_files.extend(glob.glob(str(dataset_path / '**' / ext), recursive=True))
    
    if not image_files:
        print(f"❌ No images found in {dataset_path}")
        return
    
    print(f"✅ Found {len(image_files)} images in dataset")
    
    # Select random images
    num_to_show = min(num_images, len(image_files))
    random_images = random.sample(image_files, num_to_show)
    
    print(f"\n{'='*70}")
    print(f"🎯 PREDICTIONS & GRAD-CAM FOR {num_to_show} RANDOM IMAGES")
    print(f"Dataset: {dataset_path}")
    print(f"{'='*70}\n")
    
    results = []
    
    for idx, img_path in enumerate(random_images, 1):
        print(f"\n{'-'*70}")
        print(f"Image {idx}/{num_to_show}: {pathlib.Path(img_path).name}")
        print(f"Path: {img_path}")
        print(f"{'-'*70}")
        
        try:
            # Load and preprocess image
            img = tf.io.read_file(img_path)
            img = tf.io.decode_image(img, channels=3, expand_animations=False)
            img = tf.cast(img, tf.float32) / 255.0
            img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
            img_tensor = tf.expand_dims(img, axis=0)
            
            # Make prediction
            preds = model.predict(img_tensor, verbose=0)
            pred_idx = np.argmax(preds[0])
            confidence = preds[0][pred_idx]
            
            # Get top 3 predictions
            top_3_indices = np.argsort(preds[0])[-3:][::-1]
            
            print(f"🎯 Top Predictions:")
            print(f"   1. {classes[top_3_indices[0]]}: {preds[0][top_3_indices[0]]*100:.2f}%")
            print(f"   2. {classes[top_3_indices[1]]}: {preds[0][top_3_indices[1]]*100:.2f}%")
            print(f"   3. {classes[top_3_indices[2]]}: {preds[0][top_3_indices[2]]*100:.2f}%")
            
            results.append({
                'path': img_path,
                'predicted_class': classes[pred_idx],
                'confidence': float(confidence),
                'pred_idx': pred_idx,
                'image_tensor': img_tensor
            })
            
            # Visualize with Grad-CAM
            visualize_gradcam_proper(img_tensor, model, true_label_idx=None)
            
        except Exception as e:
            print(f"❌ Error processing image: {e}")
            continue
    
    print(f"\n{'='*70}")
    print(f"✅ SUMMARY: Processed {len(results)} images successfully")
    print(f"{'='*70}\n")
    
    return results


# ====================================================================================
# USAGE EXAMPLES - Uncomment and modify the path to use with your dataset
# ====================================================================================
# 
# Example 1: Single disease class folder
# results = predict_and_gradcam_custom_dataset(
#     "/path/to/your/Anthracnose/folder",  # ← CHANGE THIS TO YOUR DATASET PATH
#     best_model,
#     num_images=5
# )
#
# Example 2: Multiple classes in a directory
# results = predict_and_gradcam_custom_dataset(
#     "/path/to/your/disease/images",  # ← CHANGE THIS TO YOUR DATASET PATH
#     best_model,
#     num_images=5
# )
#
# Example 3: Full dataset path
# results = predict_and_gradcam_custom_dataset(
#     "/kaggle/input/datasets/harshulbathamdev/mango-leaf/Mango_leaf_D/Healthy",
#     best_model,
#     num_images=5
# )
#
# To run, replace the path above and execute the cell below

# ==========================================
# EXECUTE THIS CELL WITH YOUR CUSTOM DATASET PATH
# ==========================================
# Uncomment and modify the path below, then run:

CUSTOM_DATASET_PATH = "/kaggle/input/datasets/brazendetective58/mango-leaf/test/test/Anthracnose"  # ← REPLACE THIS WITH YOUR ACTUAL PATH
results = predict_and_gradcam_custom_dataset(CUSTOM_DATASET_PATH, best_model, num_images=5)